## Decorator
- 함수가 다른 함수를 불러온다

### 1. 수작업

In [ ]:
collected_functions = []  # 수집된 함수들

def collect(func):          # "수집하다" — 데코레이터의 동작을 직관적으로 표현
    print(f'collecting: {func.__name__}')
    collected_functions.append(func)
    return func

def f1():
    print('running f1()')

collect(f1)

collecting: f1


<function __main__.f1()>

In [ ]:
collected_functions

[<function __main__.f1()>]

In [ ]:
def f2():
    print('running f2()')

collect(f2)

collecting: f2


<function __main__.f2()>

## 2. Decorator(@)

In [ ]:
registry = [] # 함수들이 저장될 레지스트리 (리스트)

def register(func):
    # 모듈 로드 시 이 부분이 즉시 실행됩니다.
    print(f'running register({func})')
    registry.append(func)
    return func # 장식된 함수를 그대로 반환합니다.

@register
def f1():
    print('running f1()')

@register
def f2():
    print('running f2()')

@register
def f3():
    print('running f3()')

print(f'Registry after loading module: {registry}')

running register(<function f1 at 0x7b0d729db380>)
running register(<function f2 at 0x7b0d729db100>)
running register(<function f3 at 0x7b0d729da3e0>)
Registry after loading module: [<function f1 at 0x7b0d729db380>, <function f2 at 0x7b0d729db100>, <function f3 at 0x7b0d729da3e0>]


PYDANTIC

In [ ]:
import dataclasses

## 1. dataclass 예제
# ----------------------------------------------------

@dataclasses.dataclass
class DataUser:
    """타입 힌트를 사용하지만, 런타임 유효성 검사는 없음."""
    id: int
    name: str
    is_active: bool = True

print("="*15 + " dataclass 결과 " + "="*15)

# (1) 올바른 데이터
user_data = DataUser(id=1, name="Alice")
print(f"✅ 유효한 데이터 (정상): {user_data}")

# (2) 유효하지 않은 데이터 (str 대신 int를 할당)
# dataclass는 오류를 발생시키지 않고 값을 그대로 저장함 (타입 검사 X)
invalid_user = DataUser(id=2, name=123)
print(f"⚠️ 유효하지 않은 데이터 (저장됨): {invalid_user}")

=============== dataclass 결과 ===============
✅ 유효한 데이터 (정상): DataUser(id=1, name='Alice', is_active=True)
⚠️ 유효하지 않은 데이터 (저장됨): DataUser(id=2, name=123, is_active=True)


## 클래스 타입과 Pydantic (FastAPI의 핵심)

FastAPI에서는 사용자가 정의한 클래스 자체를 타입으로 사용합니다.

특히 데이터 검증 라이브러리인 Pydantic 모델을 타입 힌트로 지정하면, FastAPI가 자동으로 JSON 데이터를 파싱하고 검증해줍니다.

In [ ]:
from pydantic import BaseModel

class Item(BaseModel):
    name: str
    price: float
    is_offer: bool | None = None

# FastAPI에서 이런 식으로 사용하게 됩니다.
def create_item(item: Item):
    return {"item_name": item.name, "item_price": item.price}

In [ ]:
create_item(item=Item(name="Foo", price=50.2))

{'item_name': 'Foo', 'item_price': 50.2}

In [ ]:
from pydantic import ValidationError

## 2. Pydantic BaseModel 예제
# ----------------------------------------------------

class PydanticUser(BaseModel):
    """타입 힌트를 사용하며, 런타임에 유효성 검사 및 강제 변환 수행."""
    id: int
    name: str
    is_active: bool = True

print("\n" + "="*15 + " Pydantic 결과 " + "="*15)

# (1) 올바른 데이터
p_user_data = PydanticUser(id=1, name="Bob")
print(f"✅ 유효한 데이터 (정상): {p_user_data}")

# (2) 타입 강제 변환 (str '2'가 int로 자동 변환됨)
p_user_coerce = PydanticUser(id='2', name="Charlie")
print(f"🔄 타입 강제 변환 (정상): {p_user_coerce}")

# (3) 유효성 검사 실패 (유효한 bool 값이 아님)
try:
    PydanticUser(id=3, name="David", is_active="not_a_bool")
except ValidationError as e:
    print(f"❌ 유효성 검사 실패: \n{e.errors()[0]}")


=============== Pydantic 결과 ===============
✅ 유효한 데이터 (정상): id=1 name='Bob' is_active=True
🔄 타입 강제 변환 (정상): id=2 name='Charlie' is_active=True
❌ 유효성 검사 실패: 
{'type': 'bool_parsing', 'loc': ('is_active',), 'msg': 'Input should be a valid boolean, unable to interpret input', 'input': 'not_a_bool', 'url': 'https://errors.pydantic.dev/2.12/v/bool_parsing'}


## 🐍 [Python 타입 힌트 (Type Hints) ](https://docs.python.org/3/library/typing.html)

- 타입 힌트만 제대로 이해해도 FastAPI 코드의 80%를 읽을 수 있음

- 기존 파이썬이 "실행해보기 전까진 뭐가 들어올지 모르는" 방식이었다면, 타입 힌트는 "여기에 뭐가 들어올지 미리 써두는" 약속

## 1. 변수와 함수에서의 기본 선언

가장 기초적인 방식 : 변수 이름 뒤에 콜론(:)을 붙여 타입을 명시

변수: name: str = "Gemini"

함수 매개변수와 반환값: def 함수명(인자: 타입) -> 반환타입:

In [ ]:
def greeting(name: str) -> str:
    return f"Hello, {name}"

# name은 문자열이어야 하고, 이 함수는 문자열을 반환한다는 뜻

greeting(name= 'Gemini')

'Hello, Gemini'

In [ ]:
def surface_area_of_cube(edge_length: float) -> str:
    return f"The surface area of the cube is {6 * edge_length ** 2}."

In [ ]:
surface_area_of_cube(edge_length = 4.00)

'The surface area of the cube is 96.0.'

In [ ]:
surface_area_of_cube(4.00)

'The surface area of the cube is 96.0.'

In [ ]:
res = surface_area_of_cube(4.00)
type(res)

str

## 2. 컬렉션 타입 (List, Dict, Tuple)

단순한 타입을 넘어 리스트 안에 무엇이 들어있는지도 명시할 수 있습니다. Python 3.9 버전부터는 기본 list, dict 키워드를 그대로 사용합니다

In [ ]:
type Vector = list[float]

def scale(scalar: float, vector: Vector) -> Vector:
    return [scalar * num for num in vector]

# passes type checking; a list of floats qualifies as a Vector.
new_vector = scale(2.0, [1.0, -4.2, 5.4])

In [ ]:
new_vector

[2.0, -8.4, 10.8]

In [ ]:
# 문자열만 담긴 리스트
names: list[str] = ["Alice", "Bob", "Charlie"]

# 키는 문자열, 값은 정수인 딕셔너리
user_scores: dict[str, int] = {"Alice": 100, "Bob": 90}

# 요소의 순서와 타입이 고정된 튜플
user_info: tuple[int, str] = (1, "Admin")

In [ ]:
names

['Alice', 'Bob', 'Charlie']

In [ ]:
user_scores

{'Alice': 100, 'Bob': 90}

In [ ]:
user_info

(1, 'Admin')

## 3. Optional과 Union (선택적 타입)

FastAPI에서 가장 많이 쓰이는 패턴입니다. 값이 있을 수도 있고 없을 수도(None) 있는 상황을 표현

Union: 여러 타입 중 하나일 수 있음

Optional: 해당 타입이거나 None임 (사실 Union[타입, None]의 줄임표현입니다.)

In [ ]:
from typing import Union

# Python 3.10+ 권장 방식 (| 기호 사용)
def get_item(item_id: int, description: str | None = None):
    return {"id": item_id, "desc": description}

# 3.10 미만 버전 방식
def get_user(user_id: Union[int, str]):
    # user_id 가 숫자일 수도, 문자열일 수도 있음
    return {"id": user_id}

In [ ]:
get_item(item_id = 1)

{'id': 1, 'desc': None}

In [ ]:
get_item(item_id = 1, description = 'test')


{'id': 1, 'desc': 'test'}

In [ ]:
get_user(user_id = 1)

{'id': 1}

In [ ]:
get_user(user_id = '1')

{'id': '1'}

In [ ]:
from typing import Optional

# Optional 및 Union
def greeting(name: str, enthusiasm: Optional[int] = None) -> str:
    # Optional[int]는 Union[int, None]과 동일
    if enthusiasm is None:
        return f"Hello, {name}."
    return f"Hello, {name}!" * enthusiasm

In [ ]:
greeting_result = greeting("Alice", 3)
print(greeting_result)

Hello, Alice!Hello, Alice!Hello, Alice!


In [ ]:
from typing import List, Tuple
from collections.abc import Mapping

# 제네릭 컬렉션 및 고정 길이/가변 길이 튜플
def process_records(data: List[Tuple[str, int]],                # 리스트에 str과 int 튜플 포함
                    settings: Mapping[str, Union[str, int]],    # dict 대신 Mapping ABC 사용
                    coordinates: Tuple[float, float, float]     # 고정 길이 3차원 좌표
                    ) -> Tuple[int, ...]:                           # 가변 길이 정수 튜플 (int, int, ...) 반환

    """레코드를 처리하고, 처리된 항목 수와 좌표를 반환합니다."""
    print(f"데이터: {data}, 데이터 타입 {type(data)}")
    processed_count = len(data)
    print(f"처리된 항목 수: {processed_count}")
    print(f"설정: {settings}, 설정 타입: {type(settings)}")
    print(f"좌표: {coordinates}, 좌표 타입: {type(coordinates)}")
    # Mapping 사용으로 dict, OrderedDict 등 다양한 맵핑 타입 허용

    # 반환 타입: 가변 길이 튜플 (여기서는 간단히 정수 카운트를 반환)
    return (processed_count)

records = [("item1", 10), ("item2", 20)]
settings = {"mode": "fast", "retry": 3}
coordinates = (1.0, 2.0, 3.0)

In [ ]:
processed = process_records(records, settings, coordinates)

데이터: [('item1', 10), ('item2', 20)], 데이터 타입 <class 'list'>
처리된 항목 수: 2
설정: {'mode': 'fast', 'retry': 3}, 설정 타입: <class 'dict'>
좌표: (1.0, 2.0, 3.0), 좌표 타입: <class 'tuple'>


## **Python 데이터베이스 강의**
SQLite3 완전 정복 — 연결부터 CRUD까지

SQLite3는 별도 서버 없이 파일 하나로 동작하는 경량 데이터베이스로 Python 표준 라이브러리에 포함되어 있어 별도 설치가 필요 없습니다.

데이터베이스의 모든 조작은 CRUD 네 가지로 요약됩니다.
<img src="https://media.licdn.com/dms/image/v2/D5612AQHMhCFuMX-R8w/article-cover_image-shrink_720_1280/article-cover_image-shrink_720_1280/0/1714631128138?e=2147483647&v=beta&t=2T8DQHZafA1NHndeBmV_XmyCvZzEg5wi6IhhgbN69Gw" width=800 height=300>


## ① DB 초기화 — init_db()

데이터베이스 파일에 연결하고, 테이블을 생성하는 과정입니다.

sqlite3.connect()는 파일이 없으면 자동으로 생성합니다. cursor는 SQL 명령을 실행하는 객체입니다.

In [ ]:
import sqlite3
import os

DB_NAME = "test_sqlite3.db"

def init_db():
    # 이전 파일이 있으면 삭제 (초기화)
    if os.path.exists(DB_NAME):
        os.remove(DB_NAME)

    # 1. DB 연결 — 파일이 없으면 자동 생성
    conn = sqlite3.connect(DB_NAME)

    # 2. Cursor 생성 — SQL 실행 담당 객체
    cursor = conn.cursor()

    # 3. 테이블 생성 (CREATE TABLE)
    cursor.execute("""
        CREATE TABLE users (
            id       INTEGER PRIMARY KEY,
            name     TEXT NOT NULL,
            fullname TEXT
        )
    """)

    conn.commit()  # 변경사항 확정 (필수!)
    return conn

## ② C — Create (데이터 삽입)
executemany()는 리스트 데이터를 한 번에 여러 행으로 삽입합니다.

In [ ]:
# Assuming init_db() from cell L-3q6dEoKLbT has been defined and is in scope.
# Re-initialize conn and cursor for the new test_sqlite3.db database.

conn = init_db() # Call init_db to get a fresh connection to test_sqlite3.db
cursor = conn.cursor() # Get a cursor for this new connection

users_data = [
    ('spongebob', 'Spongebob Squarepants'),
    ('patrick',   'Patrick Star'),
    ('squidward', 'Squidward Tentacles'),
]

# executemany: 리스트를 반복하며 여러 행 삽입
cursor.executemany(
    "INSERT INTO users (name, fullname) VALUES (?, ?)",
    users_data
)
conn.commit()
print(f"{len(users_data)}개의 사용자 데이터 삽입 완료.")

3개의 사용자 데이터 삽입 완료.


## ③ R — Read (데이터 조회)

fetchall()은 모든 행을 리스트로, fetchone()은 첫 번째 행 하나만 반환합니다.

결과는 튜플(tuple) 형식이며, 인덱스로 각 컬럼에 접근합니다.


In [ ]:
# 전체 조회 — fetchall()
cursor.execute("SELECT id, name, fullname FROM users")
rows = cursor.fetchall()

for row in rows:
    print(f"ID: {row[0]}, Name: {row[1]}, FullName: {row[2]}")

ID: 1, Name: spongebob, FullName: Spongebob Squarepants
ID: 2, Name: patrick, FullName: Patrick Star
ID: 3, Name: squidward, FullName: Squidward Tentacles


In [ ]:
# 조건 조회 — WHERE + fetchone()
cursor.execute(
    "SELECT * FROM users WHERE name = ?",
    ('patrick',)   # 튜플 주의: 쉼표 필수
)
patrick = cursor.fetchone()
print(patrick)

(2, 'patrick', 'Patrick Star')


## ④ U — Update (데이터 수정)

UPDATE ... SET ... WHERE 구문으로 특정 행의 값을 수정합니다.

WHERE 조건을 빠뜨리면 테이블 전체가 수정되니 항상 확인하세요.

In [ ]:
new_fullname = 'S. Squarepants'

cursor.execute(
    "UPDATE users SET fullname = ? WHERE name = ?",
    (new_fullname, 'spongebob')
)
conn.commit()

# 변경 확인
cursor.execute("SELECT * FROM users WHERE name = 'spongebob'")
print(cursor.fetchone())

(1, 'spongebob', 'S. Squarepants')


## ⑤ D — Delete (데이터 삭제)

DELETE FROM ... WHERE로 조건에 맞는 행을 삭제합니다.

WHERE 없이 실행하면 테이블의 모든 데이터가 삭제됩니다.

In [ ]:
cursor.execute(
    "DELETE FROM users WHERE name = ?",
    ('squidward',)
)
conn.commit()

# 최종 데이터 확인
cursor.execute("SELECT * FROM users")
for row in cursor.fetchall():
    print(row)

(1, 'spongebob', 'S. Squarepants')
(2, 'patrick', 'Patrick Star')
